# Edge sensitivity and Shapley results

This notebook reads the fixed-weight edge ablation and sampled Shapley outputs for the `replace_000` and `replace_100` models.

- **Edge removal**: disables one existing C?C connection, with the trained parameters held fixed. The drop is baseline metric minus the metric after removal.
- **Shapley**: averages each edge's marginal change in the chosen metric over sampled edge orderings.
- **Retraining selections**: ranks validation-set removals using `SELECTION_METRIC`, then writes the top N edges for each model to `selected_edges_for_retraining.csv`. The final test split is displayed for reporting, not used to choose edges.

Change `DATASET` to `celeba` or `cfmnist` and adjust `SELECTION_METRIC` below.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "experiments").exists() and (PROJECT_ROOT.parent / "experiments").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET = "celeba"  # "celeba" or "cfmnist"
ANALYSIS_CONFIG = PROJECT_ROOT / "all_configs" / "analysis" / "edge_sensitivity" / f"{DATASET}_replace_000_vs_replace_100.yaml"
analysis_config = yaml.safe_load(ANALYSIS_CONFIG.read_text(encoding="utf-8"))
model_specs = {m["name"]: m for m in analysis_config["models"]}
SELECTION_METRIC = analysis_config.get("selection_metric", "task_accuracy")  # can override here, e.g. concept_accuracy_003
TOP_N = int(analysis_config.get("top_edges_per_model", 3))
RESULTS_DIR = PROJECT_ROOT / "experiments" / "edge_sensitivity" / DATASET
SELECTION_CSV = RESULTS_DIR / "selected_edges_for_retraining.csv"

coalitions = pd.read_csv(RESULTS_DIR / "coalition_metrics.csv")
shapley = pd.read_csv(RESULTS_DIR / "shapley_values.csv")
edges = pd.read_csv(RESULTS_DIR / "edge_list.csv")

# Reproducible paper-style exports for edge sensitivity and Shapley analysis.
from matplotlib.colors import LinearSegmentedColormap
REPORT_DIR = PROJECT_ROOT / "report"
CSV_REPORT_DIR = REPORT_DIR / "a_metric_sens_csv"
FIGURE_REPORT_DIR = REPORT_DIR / "a_metric_sens_figures"
CSV_REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
LATEX_REPORT_PATH = REPORT_DIR / "a_metric_sens_latex.tex"
LATEX_FIGURES = []
PAPER_CMAP = LinearSegmentedColormap.from_list(
    "blue_grey_purple", ["#3B6FB6", "#D8D9DE", "#6542A6"]
)
EDGE_PALETTE = [
    "#6542A6", "#3B6FB6", "#737782", "#8B6CC1", "#5793CC", "#9A9DA5",
    "#4D3A80", "#2D5F91", "#B2B4BA", "#A084CC", "#78A9D6", "#5C626E",
    "#7B63A8", "#466F9F", "#858A94", "#B39AD8", "#91B9DD", "#C4C6CB",
]
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["STIXGeneral", "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 10,
    "figure.dpi": 130,
    "savefig.dpi": 300,
    "axes.edgecolor": "#555963",
    "text.color": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
})

def save_paper_figure(fig, stem, caption):
    pdf_path = FIGURE_REPORT_DIR / f"{stem}.pdf"
    png_path = FIGURE_REPORT_DIR / f"{stem}.png"
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, bbox_inches="tight", dpi=300)
    LATEX_FIGURES[:] = [item for item in LATEX_FIGURES if item["stem"] != stem]
    LATEX_FIGURES.append({"stem": stem, "pdf": pdf_path.relative_to(PROJECT_ROOT).as_posix(), "caption": caption})
    return pdf_path, png_path

def write_latex_report():
    blocks = [
        "% Auto-generated by notebook/edge_sensitivity_shapley_analysis.ipynb.",
        "% Compile from the repository root; requires \\usepackage{graphicx}.",
        "",
    ]
    for item in LATEX_FIGURES:
        label = "fig:" + item["stem"].replace("_", "-")
        blocks.extend([
            "\\begin{figure}[t]", "\\centering",
            f"\\includegraphics[width=0.95\\linewidth,height=0.88\\textheight,keepaspectratio]{{{item['pdf']}}}",
            f"\\caption{{{item['caption']}}}", f"\\label{{{label}}}",
            "\\end{figure}", "",
        ])
    LATEX_REPORT_PATH.write_text("\n".join(blocks) + "\n", encoding="utf-8")
    print("Wrote LaTeX figure snippet:", LATEX_REPORT_PATH)

print("Loaded", DATASET, "from", RESULTS_DIR)
print("Models:", sorted(edges.model.unique()))
print("Available metric examples:", sorted(coalitions.metric.unique())[:30])
# Original CREAM run used as a reference (from the existing topology-density analysis).
ORIGINAL_RUNS = {
    "celeba": PROJECT_ROOT / "experiments/CelebA/train_cbm/Standard_CelebA/CREAM/CREAM_best_celeba",
    "cfmnist": PROJECT_ROOT / "experiments/Complete_Concept_FMNIST/train_cbm/Standard_FashionMNIST/CREAM/CREAM_best_cfmnist",
}
ORIGINAL_RUN_ROOT = ORIGINAL_RUNS[DATASET]
_original_metric_files = sorted(ORIGINAL_RUN_ROOT.glob("**/last_metrics/*.csv")) if ORIGINAL_RUN_ROOT.exists() else []
ORIGINAL_METRICS_PATH = _original_metric_files[-1] if _original_metric_files else None
_original_intervention_files = sorted(ORIGINAL_RUN_ROOT.glob("**/intervention_results.csv")) if ORIGINAL_RUN_ROOT.exists() else []
ORIGINAL_INTERVENTIONS_PATH = _original_intervention_files[-1] if _original_intervention_files else None
print("Original CREAM reference:", ORIGINAL_RUN_ROOT)
print("Original CREAM metrics:", ORIGINAL_METRICS_PATH or "not found")
print("Original CREAM intervention CSV:", ORIGINAL_INTERVENTIONS_PATH or "not found")


## 1. Baseline predictions and intervention performance\n\nThe table compares the two topology models; the original CREAM test metrics and saved intervention curve are added when their result files are present. The next plot shows test task accuracy as concepts are intervened on. This intervention curve is separate from the edge-removal heatmap below.\n

In [ ]:
baseline = coalitions[(coalitions.coalition_type == "baseline") & (coalitions.split.isin(["val", "test"]))].copy()
base_pivot = baseline.pivot_table(index=["metric"], columns=["model", "split"], values="metric_value")
display(base_pivot.round(4))
base_pivot.reset_index().to_csv(CSV_REPORT_DIR / "baseline_topology_metrics.csv", index=False)

# Original CREAM has no edge-removal rows, so compare it as a baseline reference.
if ORIGINAL_METRICS_PATH is not None:
    original_row = pd.read_csv(ORIGINAL_METRICS_PATH).iloc[0]
    original_values = {"task_accuracy": original_row.get("test_task_accuracy"),
                       "concept_accuracy": original_row.get("test_concept_accuracy")}
    original_table = pd.DataFrame({"Original CREAM (test)": original_values})
    print("Original CREAM baseline metrics (from its saved last_metrics CSV):")
    display(original_table.round(4))
    original_table.rename_axis("metric").reset_index().to_csv(CSV_REPORT_DIR / "baseline_original_cream_metrics.csv", index=False)

# Plot test task-accuracy intervention curves for replace_000 and replace_100.
# Add the curve saved by the original CREAM run when that file is present.
curve = baseline[baseline.metric.str.startswith("intervention_task_accuracy_") & (baseline.split == "test")].copy()
if not curve.empty:
    parts = curve.metric.str.extract(r"intervention_task_accuracy_(individual|group)_(\d+)")
    curve["intervention_mode"] = parts[0]
    curve["num_interventions"] = parts[1].astype(int)

original_curve = pd.DataFrame()
if ORIGINAL_INTERVENTIONS_PATH is not None:
    original_curve = pd.read_csv(ORIGINAL_INTERVENTIONS_PATH)
    if not original_curve.empty and "num_interventions" in original_curve:
        task_col = next((c for c in ["test_task_accuracy", "task_accuracy", "test_accuracy", "accuracy"] if c in original_curve), None)
        if task_col:
            original_curve["curve_accuracy"] = pd.to_numeric(original_curve[task_col], errors="coerce")
            original_curve["intervention_mode"] = original_curve.get("group_interventions", False).map(
                lambda x: "group" if str(x).strip().lower() in ("true", "1") else "individual"
            ) if hasattr(original_curve.get("group_interventions", False), "map") else "individual"

curve_export_rows = []
if not curve.empty:
    curve_export_rows.extend(curve[["model", "split", "intervention_mode", "num_interventions", "metric_value"]]
                             .rename(columns={"metric_value": "task_accuracy"}).to_dict("records"))
if not original_curve.empty and "curve_accuracy" in original_curve:
    curve_export_rows.extend(original_curve[["intervention_mode", "num_interventions", "curve_accuracy"]]
                             .rename(columns={"curve_accuracy": "task_accuracy"})
                             .assign(model="Original CREAM", split="test")
                             [["model", "split", "intervention_mode", "num_interventions", "task_accuracy"]].to_dict("records"))
pd.DataFrame(curve_export_rows).to_csv(CSV_REPORT_DIR / "test_intervention_curves.csv", index=False)

modes = sorted(set(curve.intervention_mode.dropna().unique()) | set(original_curve.intervention_mode.dropna().unique())) if not curve.empty or not original_curve.empty else []
for mode in modes:
    fig, ax = plt.subplots(figsize=(7.2, 3.5), constrained_layout=True)
    current = curve[curve.intervention_mode == mode] if not curve.empty else pd.DataFrame()
    for model, frame in current.groupby("model"):
        ax.plot(frame.num_interventions, frame.metric_value, marker="o", linewidth=1.8,
                color={"replace_000": "#6542A6", "replace_100": "#3B6FB6"}.get(model, "#737782"), label=model)
    if not original_curve.empty:
        old = original_curve[original_curve.intervention_mode == mode].copy()
        if not old.empty:
            ax.plot(old.num_interventions, old.curve_accuracy, marker="s", linestyle="--", color="#737782", label="Original CREAM")
    ax.set(title=f"Test task accuracy under {mode} interventions", xlabel="Number of intervened concepts", ylabel="Task accuracy")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="both", colors="black", labelsize=12)
    ax.title.set_color("black")
    ax.xaxis.label.set_color("black")
    ax.yaxis.label.set_color("black")
    ax.legend()
    save_paper_figure(fig, f"{DATASET}_intervention_task_accuracy_{mode}",
                      f"Test task accuracy under {mode} interventions for Graph A, Graph B, and original CREAM when available ({DATASET}).")
    plt.show()


## 2. Which edges matter? Separate figures for each graph\n\nPrediction metrics and intervention metrics are plotted separately. Each plot is for one graph: Graph A (`replace_000`) or Graph B (`replace_100`). The horizontal color scale shows baseline validation accuracy minus accuracy after removing an edge. Positive values mean removal hurt that metric; negative values mean it improved.\n

In [ ]:
base = coalitions[(coalitions.coalition_type == "baseline")].set_index(["model", "split", "metric"]).metric_value
abl = coalitions[coalitions.coalition_type == "single_edge_removed"].copy()
abl["baseline_value"] = [base.loc[(r.model, r.split, r.metric)] for r in abl.itertuples()]
abl["metric_drop_from_removal"] = abl.baseline_value - abl.metric_value
edge_effects = abl.copy()
validation_effects = edge_effects[edge_effects.split == "val"].copy()
edge_effects.to_csv(CSV_REPORT_DIR / "edge_sensitivity_all_metrics.csv", index=False)

# Human-readable edge names use the graph's source and target concept labels.
validation_effects["edge_label"] = (
    validation_effects.source_concept.astype(str).str.replace("_", " ", regex=False)
    + " to "
    + validation_effects.target_concept.astype(str).str.replace("_", " ", regex=False)
)
selected_view = validation_effects[validation_effects.metric == SELECTION_METRIC].sort_values(
    "metric_drop_from_removal", ascending=False
)
for model, frame in selected_view.groupby("model"):
    print(f"{model} ranked by validation {SELECTION_METRIC}")
    display(frame[["edge_label", "baseline_value", "metric_value", "metric_drop_from_removal"]]
            .head(15).round(4))

# Separate prediction metrics from intervention metrics. Each category gets its
# own compact figure with Graph A above Graph B.
model_order = [m for m in ["replace_000", "replace_100"] if m in validation_effects.model.unique()]
all_metric_order = list(validation_effects.metric.drop_duplicates())
all_metric_order = sorted(all_metric_order, key=lambda m: (
    0 if m == "task_accuracy" else 1 if m == "concept_accuracy" else
    2 if m.startswith("concept_accuracy_") else
    3 if m.startswith("intervention_task_auc") else
    4 if m.startswith("intervention_concept_auc") else 5, m
))
concept_by_index = {}
for r in edges[["source_index", "source_concept"]].drop_duplicates().itertuples(index=False):
    concept_by_index[int(r.source_index)] = str(r.source_concept).replace("_", " ")
for r in edges[["target_index", "target_concept"]].drop_duplicates().itertuples(index=False):
    concept_by_index[int(r.target_index)] = str(r.target_concept).replace("_", " ")

def pretty_metric(metric):
    if metric == "task_accuracy": return "TAcc"
    if metric == "concept_accuracy": return "CAcc"
    m = __import__("re").fullmatch(r"concept_accuracy_(\d+)", metric)
    if m: return "CAcc: " + concept_by_index.get(int(m.group(1)), f"Concept {m.group(1)}")
    m = __import__("re").fullmatch(r"intervention_(task|concept)_accuracy_(individual|group)_(\d+)", metric)
    if m:
        target, mode, count = m.groups()
        return f"IntAcc ({'task' if target == 'task' else 'concept'}, {mode}, n={int(count)})"
    m = __import__("re").fullmatch(r"intervention_(task|concept)_auc_(individual|group)", metric)
    if m: return f"IntAUC ({m.group(1)}, {m.group(2)})"
    return metric.replace("_", " ").title()

def plot_edge_heatmap(metrics, category, title):
    if not metrics: return
    drops = validation_effects[validation_effects.metric.isin(metrics)].metric_drop_from_removal.to_numpy(dtype=float)
    color_limit = float(np.nanmax(np.abs(drops))) if len(drops) else 1.0
    color_limit = color_limit or 1.0
    for model in model_order:
        frame = validation_effects[(validation_effects.model == model) & validation_effects.metric.isin(metrics)]
        if frame.empty: continue
        edge_order = (frame[["edge_label", "source_index", "target_index"]].drop_duplicates()
                      .sort_values(["source_index", "target_index"]).edge_label.tolist())
        matrix = (frame.pivot_table(index="metric", columns="edge_label", values="metric_drop_from_removal", aggfunc="mean")
                  .reindex(index=metrics, columns=edge_order))
        height = max(4.0, 0.30 * len(metrics) + 2.0)
        width = max(9.5, min(14, 0.75 * len(edge_order) + 4))
        fig, ax = plt.subplots(figsize=(width, height), layout="constrained")
        image = ax.imshow(matrix.to_numpy(dtype=float), aspect="auto", cmap=PAPER_CMAP,
                          vmin=-color_limit, vmax=color_limit, interpolation="nearest")
        graph_label = "Graph A" if model == "replace_000" else "Graph B"
        ax.set_title(f"{graph_label} ({model}): {title}", loc="left", color="black", fontsize=12, pad=5)
        ax.set_xticks(np.arange(len(edge_order)))
        ax.set_xticklabels([label.replace(" to ", "\nto ") for label in edge_order],
                           rotation=45, ha="right", rotation_mode="anchor", fontsize=10, color="black")
        ax.set_yticks(np.arange(len(metrics)))
        ax.set_yticklabels([pretty_metric(m) for m in metrics], fontsize=11, color="black")
        ax.set_xlabel("Removed edge: source concept to target concept", color="black", fontsize=12)
        ax.set_ylabel("Metric", color="black", fontsize=12)
        ax.tick_params(axis="both", colors="black", length=2)
        if len(metrics) <= 16 and len(edge_order) <= 12:
            for i in range(matrix.shape[0]):
                for j in range(matrix.shape[1]):
                    value = matrix.iat[i, j]
                    if pd.notna(value):
                        ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=9, color="black")
        cbar = fig.colorbar(image, ax=ax, orientation="horizontal", pad=0.24, fraction=0.08, aspect=35)
        cbar.set_label("Baseline minus after edge removal (positive means removal hurts)", color="black", fontsize=11)
        cbar.ax.tick_params(colors="black", labelsize=10)
        stem = f"{DATASET}_edge_sensitivity_{category}_{model}"
        save_paper_figure(fig, stem,
                          f"{graph_label} ({model}) {title.lower()} for {DATASET}; positive values mean edge removal reduced performance.")
        plt.show()

prediction_metrics = [m for m in all_metric_order if not m.startswith("intervention_")]
intervention_metrics = [m for m in all_metric_order if m.startswith("intervention_")]
plot_edge_heatmap(prediction_metrics, "prediction_metrics", "Edge sensitivity: prediction metrics")
plot_edge_heatmap(intervention_metrics, "intervention_metrics", "Edge sensitivity: intervention metrics")


## 3. Shapley contributions: prediction and intervention metrics\n\nThe table ranks edges for the configured selection metric. Each graph has separate horizontal plots for prediction metrics and intervention metrics. Rows name the metric; colored dots identify source-to-target edges; the x-position is Shapley value and horizontal whiskers show permutation standard error. Positive values mean enabling that edge improves the metric on average across sampled coalitions.\n

In [ ]:
# Keep the ranked table for the configured selection metric, then plot every recorded metric.
shapley.to_csv(CSV_REPORT_DIR / "shapley_all_metrics.csv", index=False)
shapley_metric = SELECTION_METRIC
shapley_part = shapley[shapley.metric == shapley_metric].copy()
for model, frame in shapley_part.groupby("model"):
    frame = frame.sort_values("shapley_value", ascending=False)
    print(f"{model} ranked by Shapley value for {shapley_metric}")
    display(frame[["source_concept", "target_concept", "shapley_value", "permutation_se", "n_permutations"]].head(20).round(5))

# Readable names for metrics, including individual-concept accuracy and intervention metrics.
concept_by_index = {}
for r in edges[["source_index", "source_concept"]].drop_duplicates().itertuples(index=False):
    concept_by_index[int(r.source_index)] = str(r.source_concept).replace("_", " ")
for r in edges[["target_index", "target_concept"]].drop_duplicates().itertuples(index=False):
    concept_by_index[int(r.target_index)] = str(r.target_concept).replace("_", " ")

def shapley_metric_label(metric):
    if metric == "task_accuracy": return "TAcc"
    if metric == "concept_accuracy": return "CAcc"
    m = __import__("re").fullmatch(r"concept_accuracy_(\d+)", metric)
    if m: return "CAcc: " + concept_by_index.get(int(m.group(1)), f"Concept {m.group(1)}")
    m = __import__("re").fullmatch(r"intervention_(task|concept)_accuracy_(individual|group)_(\d+)", metric)
    if m:
        target, mode, count = m.groups()
        target = "task" if target == "task" else "concept"
        return f"IntAcc ({target}, {mode}, n={int(count)})"
    m = __import__("re").fullmatch(r"intervention_(task|concept)_auc_(individual|group)", metric)
    if m:
        target, mode = m.groups()
        return f"IntAUC ({target}, {mode})"
    return metric.replace("_", " ").title()

metric_order = list(shapley.metric.drop_duplicates())
metric_order = sorted(metric_order, key=lambda m: (
    0 if m == "task_accuracy" else 1 if m == "concept_accuracy" else
    2 if m.startswith("concept_accuracy_") else
    3 if m.startswith("intervention_task_auc") else
    4 if m.startswith("intervention_concept_auc") else 5, m
))

all_metric_order = list(shapley.metric.drop_duplicates())
all_metric_order = sorted(all_metric_order, key=lambda m: (
    0 if m == "task_accuracy" else 1 if m == "concept_accuracy" else
    2 if m.startswith("concept_accuracy_") else
    3 if m.startswith("intervention_task_auc") else
    4 if m.startswith("intervention_concept_auc") else 5, m
))
metric_groups = {
    "prediction_metrics": [m for m in all_metric_order if not m.startswith("intervention_")],
    "intervention_metrics": [m for m in all_metric_order if m.startswith("intervention_")],
}

# One horizontal plot per model and metric family. X is Shapley value; rows are
# metrics. Colors identify the edge (source concept to target concept).
for model in ["replace_000", "replace_100"]:
    model_frame = shapley[shapley.model == model].copy()
    if model_frame.empty:
        print(f"No Shapley rows found for {model}")
        continue
    model_frame["edge_label"] = (model_frame.source_concept.astype(str).str.replace("_", " ", regex=False)
                                 + " to "
                                 + model_frame.target_concept.astype(str).str.replace("_", " ", regex=False))
    edge_order = (model_frame[["edge_label", "source_index", "target_index"]].drop_duplicates()
                  .sort_values(["source_index", "target_index"]).edge_label.tolist())
    palette = [EDGE_PALETTE[i % len(EDGE_PALETTE)] for i in range(len(edge_order))]
    for family, metrics in metric_groups.items():
        if not metrics: continue
        frame = model_frame[model_frame.metric.isin(metrics)].copy()
        fig_height = max(4.0, 0.34 * len(metrics) + 1.5)
        fig, ax = plt.subplots(figsize=(9.2, fig_height), layout="constrained")
        offsets = np.linspace(-0.28, 0.28, len(edge_order)) if len(edge_order) > 1 else np.array([0.0])
        for edge_idx, edge_label in enumerate(edge_order):
            edge_rows = frame[frame.edge_label == edge_label]
            y_by_metric = {metric: i for i, metric in enumerate(metrics)}
            y = np.array([y_by_metric[m] + offsets[edge_idx] for m in edge_rows.metric])
            ax.errorbar(edge_rows.shapley_value.to_numpy(dtype=float), y,
                        xerr=edge_rows.permutation_se.to_numpy(dtype=float), fmt="o",
                        markersize=4, capsize=2, linewidth=0.8, color=palette[edge_idx], label=edge_label)
        ax.axvline(0, color="#555963", linewidth=0.8)
        ax.set_yticks(np.arange(len(metrics)))
        ax.set_yticklabels([shapley_metric_label(m) for m in metrics], color="black", fontsize=10)
        ax.invert_yaxis()
        ax.set_xlabel("Shapley value (accuracy gain from enabling the edge)", color="black", fontsize=11)
        graph_label = "Graph A" if model == "replace_000" else "Graph B"
        family_label = "intervention metrics" if family == "intervention_metrics" else "prediction metrics"
        ax.set_title(f"{graph_label} ({model}): {family_label}", loc="left", color="black", fontsize=12)
        ax.tick_params(axis="both", colors="black", labelsize=12, length=2)
        ax.grid(axis="x", color="#D8D9DE", linewidth=0.6)
        ax.legend(title="Edge: source to target concept", loc="upper center", bbox_to_anchor=(0.5, -0.12),
                  ncol=min(3, max(1, len(edge_order))), frameon=False, fontsize=8, title_fontsize=9)
        save_paper_figure(fig, f"{DATASET}_shapley_{family}_{model}",
                          f"Shapley values for {family_label} for {graph_label} ({model}) on {DATASET}; colors identify source-to-target concept edges and whiskers show permutation standard error.")
        plt.show()

write_latex_report()
print("Saved CSV tables under:", CSV_REPORT_DIR)
print("Saved PDF and PNG figures under:", FIGURE_REPORT_DIR)


## 4. Retraining results: did the model compensate?

For each edge ranked on validation data, compare three test results: the original trained topology, the same checkpoint with the edge removed only at evaluation, and the model retrained with that edge absent. The retraining gain is `retrained - edge removed at evaluation`; the remaining gap is `original - retrained`. Positive gain means training recovered some lost performance. The notebook loads every matching seed result it can find and flags candidates whose retraining result is not present yet.


In [ ]:
import re
from pathlib import Path

# Find completed edge-removal retraining runs using the generated experiment layout.
retraining_records = []
retraining_files = []
retraining_intervention_files = []
base_metric_rows = coalitions[(coalitions.coalition_type == "baseline") & (coalitions.split == "test")]
removed_metric_rows = coalitions[(coalitions.coalition_type == "single_edge_removed") & (coalitions.split == "test")]

for model_name, spec in model_specs.items():
    cfg_path = Path(spec["config"])
    if not cfg_path.is_absolute(): cfg_path = PROJECT_ROOT / cfg_path
    train_cfg = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))
    root = Path(train_cfg["paths"].get("default_root_dir", "./experiments"))
    if not root.is_absolute(): root = PROJECT_ROOT / root
    generated_config_folder = analysis_config["dataset"].lower().replace("_", "")
    run_root = (root / train_cfg["dataset_name"] / train_cfg["mode"] / train_cfg["model_name"]
                / generated_config_folder / "edge_sensitivity" / model_name)
    if not run_root.exists():
        print(f"No retraining output folder found for {model_name}: {run_root}")
        continue
    for metrics_path in sorted(run_root.glob("remove_*/seed_*/last_metrics/*.csv")):
        if metrics_path.name.endswith("_per_concept_accuracy.csv"):
            continue
        edge_folder = metrics_path.parent.parent.parent.name
        edge_id = edge_folder.removeprefix("remove_")
        seed_folder = metrics_path.parent.parent.name
        seed_match = re.fullmatch(r"seed_(\d+)", seed_folder)
        if not seed_match: continue
        seed = int(seed_match.group(1))
        metric_df = pd.read_csv(metrics_path)
        if metric_df.empty: continue
        metric_row = metric_df.iloc[0]
        retrained_values = {}
        for column, value in metric_row.items():
            if pd.isna(value): continue
            if column == "test_task_accuracy": retrained_values["task_accuracy"] = float(value)
            elif column == "test_concept_accuracy": retrained_values["concept_accuracy"] = float(value)
            else:
                match = re.fullmatch(r"test_concept_accuracy_(\d+)", str(column))
                if match: retrained_values[f"concept_accuracy_{int(match.group(1)):03d}"] = float(value)

        # CREAM writes the intervention sweep separately below the Lightning checkpoint folder.
        intervention_files = sorted(metrics_path.parents[2].glob("**/intervention_results.csv"))
        retrained_intervention_values = {}
        for intervention_path in intervention_files:
            iv = pd.read_csv(intervention_path)
            if iv.empty or "num_interventions" not in iv.columns: continue
            task_col = next((c for c in ["test_task_accuracy", "task_accuracy"] if c in iv.columns), None)
            concept_col = next((c for c in ["test_concept_accuracy", "concept_accuracy"] if c in iv.columns), None)
            if task_col is None and concept_col is None: continue
            if "group_interventions" in iv.columns:
                iv["_mode"] = iv.group_interventions.map(lambda v: "group" if str(v).strip().lower() in ("true", "1") else "individual")
            else:
                iv["_mode"] = "individual"
            for mode, mode_rows in iv.groupby("_mode"):
                mode_rows = mode_rows.sort_values("num_interventions")
                counts = pd.to_numeric(mode_rows.num_interventions, errors="coerce").to_numpy(dtype=float)
                if task_col:
                    vals = pd.to_numeric(mode_rows[task_col], errors="coerce").to_numpy(dtype=float)
                    valid = np.isfinite(counts) & np.isfinite(vals)
                    if valid.any():
                        for count, val in zip(counts[valid].astype(int), vals[valid]):
                            retrained_intervention_values[f"intervention_task_accuracy_{mode}_{count:03d}"] = float(val)
                        x = counts[valid] / max(float(counts[valid].max()), 1.0)
                        retrained_intervention_values[f"intervention_task_auc_{mode}"] = float(np.trapz(vals[valid], x)) if valid.sum() > 1 else float(vals[valid][0])
                if concept_col:
                    vals = pd.to_numeric(mode_rows[concept_col], errors="coerce").to_numpy(dtype=float)
                    valid = np.isfinite(counts) & np.isfinite(vals)
                    if valid.any():
                        for count, val in zip(counts[valid].astype(int), vals[valid]):
                            retrained_intervention_values[f"intervention_concept_accuracy_{mode}_{count:03d}"] = float(val)
                        x = counts[valid] / max(float(counts[valid].max()), 1.0)
                        retrained_intervention_values[f"intervention_concept_auc_{mode}"] = float(np.trapz(vals[valid], x)) if valid.sum() > 1 else float(vals[valid][0])
            retraining_intervention_files.append(str(intervention_path))
        retrained_values.update(retrained_intervention_values)
        retraining_records.append({"model": model_name, "edge_id": edge_id, "seed": seed,
                                  "metrics_path": str(metrics_path), "values": retrained_values})
        retraining_files.append(str(metrics_path))

print(f"Loaded {len(retraining_records)} retrained model result(s) from {len(retraining_files)} metrics CSV(s).")
if retraining_records:
    display(pd.DataFrame([{k: r[k] for k in ["model", "edge_id", "seed", "metrics_path"]} for r in retraining_records]))

# Rank every tested edge by the configured validation selection metric.
ranked_candidates = selected_view.sort_values(["model", "metric_drop_from_removal"], ascending=[True, False]).copy()
ranked_candidates["rank"] = ranked_candidates.groupby("model").cumcount() + 1
candidate_rows = []
long_rows = []
for candidate in ranked_candidates.itertuples(index=False):
    model_name, edge_id = candidate.model, candidate.edge_id
    edge_label = f"{str(candidate.source_concept).replace('_', ' ')} to {str(candidate.target_concept).replace('_', ' ')}"
    orig_vals = base_metric_rows[base_metric_rows.model == model_name].set_index("metric").metric_value.to_dict()
    ablated_rows = removed_metric_rows[(removed_metric_rows.model == model_name) & (removed_metric_rows.edge_id == edge_id)]
    ablated_vals = ablated_rows.set_index("metric").metric_value.to_dict()
    matches = [r for r in retraining_records if r["model"] == model_name and r["edge_id"] == edge_id]
    if not matches:
        matches = [{"model": model_name, "edge_id": edge_id, "seed": np.nan, "metrics_path": "", "values": {}}]
    for run in matches:
        retrain_vals = run["values"]
        all_metrics = list(dict.fromkeys(list(orig_vals) + list(ablated_vals) + list(retrain_vals)))
        for metric in all_metrics:
            original = orig_vals.get(metric, np.nan)
            eval_removed = ablated_vals.get(metric, np.nan)
            retrained = retrain_vals.get(metric, np.nan)
            gain = retrained - eval_removed if pd.notna(retrained) and pd.notna(eval_removed) else np.nan
            residual = original - retrained if pd.notna(original) and pd.notna(retrained) else np.nan
            lost = original - eval_removed if pd.notna(original) and pd.notna(eval_removed) else np.nan
            recovery = gain / lost if pd.notna(gain) and pd.notna(lost) and abs(lost) > 1e-12 else np.nan
            long_rows.append({"model": model_name, "rank": int(candidate.rank), "edge_id": edge_id,
                              "edge": edge_label, "seed": run["seed"], "metric": metric,
                              "original_model": original, "edge_removed_at_evaluation": eval_removed,
                              "retrained_without_edge": retrained, "retraining_gain": gain,
                              "remaining_gap_to_original": residual, "recovery_fraction": recovery,
                              "retrain_status": "completed" if pd.notna(retrained) else "not found",
                              "metrics_path": run["metrics_path"]})
        selection_original = orig_vals.get(SELECTION_METRIC, np.nan)
        selection_removed = ablated_vals.get(SELECTION_METRIC, np.nan)
        selection_retrained = retrain_vals.get(SELECTION_METRIC, np.nan)
        candidate_rows.append({"model": model_name, "rank": int(candidate.rank), "edge_id": edge_id,
                               "edge": edge_label, "selection_metric": SELECTION_METRIC,
                               "original_model": selection_original,
                               "edge_removed_at_evaluation": selection_removed,
                               "retrained_without_edge": selection_retrained,
                               "retraining_gain": selection_retrained - selection_removed if pd.notna(selection_retrained) and pd.notna(selection_removed) else np.nan,
                               "remaining_gap_to_original": selection_original - selection_retrained if pd.notna(selection_original) and pd.notna(selection_retrained) else np.nan,
                               "seed": run["seed"], "status": "completed" if pd.notna(selection_retrained) else "not found",
                               "metrics_path": run["metrics_path"]})

retraining_long = pd.DataFrame(long_rows)
retraining_ranked = pd.DataFrame(candidate_rows)
retraining_long.to_csv(CSV_REPORT_DIR / "retraining_compensation_all_metrics_long.csv", index=False)
retraining_ranked.to_csv(CSV_REPORT_DIR / "retraining_ranked_comparison.csv", index=False)
print("Ranked comparison: original checkpoint -> edge removed at evaluation -> retrained without edge")
display(retraining_ranked.round(4))

if not retraining_long.empty:
    # Detailed before/after table: task, overall concept, each concept, and intervention AUCs.
    key_rows = retraining_long[(retraining_long.metric.isin(["task_accuracy", "concept_accuracy"])) |
                               retraining_long.metric.str.startswith("concept_accuracy_") |
                               retraining_long.metric.str.startswith("intervention_task_accuracy_") |
                               retraining_long.metric.str.startswith("intervention_concept_accuracy_") |
                               retraining_long.metric.str.contains("_auc_")].copy()
    display(key_rows[["model", "rank", "edge", "seed", "metric", "original_model",
                      "edge_removed_at_evaluation", "retrained_without_edge", "retraining_gain",
                      "remaining_gap_to_original", "recovery_fraction", "retrain_status"]].round(4))

    # Plot recovery for prediction metrics and intervention AUCs separately, A and B in separate figures.
    plot_metrics = [m for m in key_rows.metric.drop_duplicates()]
    for model_name in ["replace_000", "replace_100"]:
        model_rows = key_rows[(key_rows.model == model_name) & (key_rows.retrain_status == "completed")]
        if model_rows.empty: continue
        for family, metrics in [("prediction", [m for m in plot_metrics if not m.startswith("intervention_")]),
                                ("intervention", [m for m in plot_metrics if m.startswith("intervention_")])]:
            metrics = [m for m in metrics if m in model_rows.metric.unique()]
            if not metrics: continue
            plot_data = model_rows[model_rows.metric.isin(metrics)].groupby(["rank", "edge_id", "edge", "metric"], as_index=False).agg(
                original_model=("original_model", "mean"), edge_removed_at_evaluation=("edge_removed_at_evaluation", "mean"),
                retrained_without_edge=("retrained_without_edge", "mean"), retraining_gain=("retraining_gain", "mean"))
            edge_order = plot_data[["rank", "edge_id", "edge"]].drop_duplicates().sort_values("rank")
            matrix = plot_data.pivot(index="metric", columns="edge_id", values="retraining_gain").reindex(index=metrics, columns=edge_order.edge_id)
            limit = float(np.nanmax(np.abs(matrix.to_numpy(dtype=float)))) if matrix.size else 1.0
            limit = limit or 1.0
            fig, ax = plt.subplots(figsize=(max(9, 0.7 * len(edge_order) + 4), max(4, 0.32 * len(metrics) + 1.6)), layout="constrained")
            im = ax.imshow(matrix.to_numpy(dtype=float), aspect="auto", cmap=PAPER_CMAP, vmin=-limit, vmax=limit, interpolation="nearest")
            ax.set_xticks(np.arange(len(edge_order)))
            ax.set_xticklabels([f"Rank {r.rank}: {r.edge}" for r in edge_order.itertuples(index=False)], rotation=45, ha="right", fontsize=9, color="black")
            ax.set_yticks(np.arange(len(metrics)))
            ax.set_yticklabels([shapley_metric_label(m) for m in metrics], fontsize=10, color="black")
            ax.set_xlabel("Ranked edge removed before retraining", fontsize=11, color="black")
            ax.set_ylabel("Metric", fontsize=11, color="black")
            ax.set_title(f"{model_name}: retraining gain ({family} metrics)", loc="left", fontsize=12, color="black")
            ax.tick_params(colors="black")
            for i in range(matrix.shape[0]):
                for j in range(matrix.shape[1]):
                    value = matrix.iat[i, j]
                    if pd.notna(value): ax.text(j, i, f"{value:.3f}", ha="center", va="center", fontsize=9, color="black")
            cbar = fig.colorbar(im, ax=ax, orientation="horizontal", pad=0.28, fraction=0.08, aspect=35)
            cbar.set_label("Retrained score minus edge-removed evaluation score", fontsize=10, color="black")
            cbar.ax.tick_params(colors="black")
            save_paper_figure(fig, f"{DATASET}_retraining_gain_{family}_{model_name}",
                              f"Recovery after retraining ranked edges for {model_name} on {DATASET}; positive means retraining improved the metric relative to removing the edge at evaluation.")
            plt.show()
    write_latex_report()


## 5. Export the top removal candidates for retraining

Selection is based on validation edge-removal effects. The output lists the top candidates per model for generating retraining configurations.


In [ ]:
if selected_view.empty:
    raise ValueError(f"No validation rows found for SELECTION_METRIC={SELECTION_METRIC!r}")
selected = (selected_view.sort_values(["model", "metric_drop_from_removal"], ascending=[True, False])
            .groupby("model", as_index=False).head(TOP_N).copy())
selected["selection_metric"] = SELECTION_METRIC
selected["selection_rank"] = selected.groupby("model")["metric_drop_from_removal"].rank(method="first", ascending=False).astype(int)
selected = selected[["model", "edge_id", "source_index", "target_index", "source_concept", "target_concept", "selection_metric", "selection_rank", "metric_drop_from_removal"]]
selected.to_csv(SELECTION_CSV, index=False)
print("Saved retraining selections:", SELECTION_CSV)
display(selected)

## Exported paper assets

Running the notebook saves analysis CSVs to `report/a_metric_sens_csv/`, PDF and PNG figures to `report/a_metric_sens_figures/`, and generates `report/a_metric_sens_latex.tex` with figure environments. The plots use a serif paper-style font and a purple, blue, and grey palette.
